# Grok 长篇小说自动写作工具

将你的角色设定和故事大纲，自动扩展为 10 章、每章约 8000 字的长篇小说。

使用前请先完成：
1. 安装依赖：pip install -r requirements.txt
2. 在下方填入你的 Grok API Key
3. 编辑 user_inputs/ 目录下的设定文件
4. 编写 chapters/ 目录下的章节大纲文件

---
## 步骤 0：配置 Grok API

In [ ]:
# 填入你的 Grok API Key
GROK_API_KEY = "xai-..."  # 替换为你的真实 Key

GROK_BASE_URL = "https://api.x.ai/v1"
GROK_MODEL = "grok-4-1-fast"
GROK_TEMPERATURE = 1.0
GROK_MAX_TOKENS = 16384

print(f"API 配置就绪，模型: {GROK_MODEL}")

---
## 步骤 0.5：快速测试 Grok API（一问一答）

In [ ]:
from grok_client import GrokClient
import config as cfg

cfg.API_BASE_URL = GROK_BASE_URL
cfg.API_MODEL = GROK_MODEL
cfg.API_TEMPERATURE = GROK_TEMPERATURE
cfg.API_MAX_TOKENS = GROK_MAX_TOKENS

client = GrokClient(
    api_key=GROK_API_KEY,
    base_url=cfg.API_BASE_URL,
    model=cfg.API_MODEL,
    temperature=cfg.API_TEMPERATURE,
    max_tokens=cfg.API_MAX_TOKENS,
    timeout=cfg.API_TIMEOUT,
    max_retries=3,
)

reply = client.chat("用一句话描述写小说的乐趣")
print(reply)

---
## 步骤 1：输入角色设定与故事内容

In [ ]:
def load_txt(path):
    with open(path, 'r', encoding='utf-8') as f:
        return f.read()

CHARACTERS = load_txt('user_inputs/characters.txt')
PLOT = load_txt('user_inputs/plot.txt')
STYLE = load_txt('user_inputs/style.txt')

print(f"角色设定: {len(CHARACTERS)} 字")
print(f"故事大纲: {len(PLOT)} 字")
print(f"写作风格: {len(STYLE)} 字")

---
## 步骤 2：初始化小说生成器

In [ ]:
from story_generator import NovelGenerator
from pdf_exporter import export_novel_to_pdf

generator = NovelGenerator(
    grok_client=client,
    characters=CHARACTERS,
    plot=PLOT,
    style=STYLE,
    output_dir="outputs",
    save_intermediates=True,
)

print("初始化完成")

---
## 步骤 3：加载章节大纲

从 chapters/ 目录读取你自行编写的章节大纲文件。

In [ ]:
chapter_outlines = generator.load_chapters_from_dir()

print(f"已加载 {len(chapter_outlines)} 章大纲")
for ch in chapter_outlines:
    print(f"  第 {ch['chapter']} 章：{ch['title']}")

---
## 步骤 4：每章拆分为 10 小段

In [ ]:
generator.generate_all_segments()

---
## 步骤 5：逐段写作全部章节

In [ ]:
generator.write_all_chapters()

print(f"\n小说全文完成！总字数：约 {len(generator.full_novel)} 字")

---
## 步骤 6：导出 PDF

In [ ]:
pdf_path = export_novel_to_pdf(
    novel_text=generator.full_novel,
    chapter_outlines=chapter_outlines,
    output_path="outputs/novel.pdf",
    title="长篇小说",
    author="Grok 自动写作",
)
print(f"PDF 已保存至: {pdf_path}")

---
## 附录：直接 Grok 聊天

In [ ]:
reply = client.chat("你的问题写在这里")
print(reply)